# Eval set semilla (M2) — continuación de S05

**SI4006 · Equipo Lawten** — Asistente de consulta de derecho laboral individual

Diez ejemplos *gold* `input → esperado` para la tarea de S05, sacados directamente del dataset real del proyecto (`data/dataset_cross_encoder.csv` + `data/diccionario_articulos.csv`), no inventados a mano. Cada `input` es una `consulta` real (extraída de una sentencia) y el `esperado` cita el/los artículo(s) que esa misma sentencia marcó como aplicable(s) (`tipo=positivo`).

Incluye 8 casos claros (una sola sentencia, un solo artículo aplicable) y 2 casos difíciles/borde:
- **T-502/00**: la consulta necesita **dos** artículos a la vez (caso multi-etiqueta real del dataset).
- **SL-1114/21**: una de las dos normas aplicables es una **convención colectiva privada** (empresa-sindicato) sin texto disponible en ninguna fuente pública — pone a prueba si el sistema reconoce ese límite en vez de inventar contenido.

In [ ]:
# Eval set semilla — 10 ejemplos reales del dataset del proyecto (data/dataset_cross_encoder.csv).
eval_set = [
    {'input': 'Trabajé desde octubre de 2023 en funciones de atención al cliente y administración, con horario de lunes a sábado. En mayo de 2024 me enteré de mi embarazo y el 28 de mayo me despidieron sin autorización del Ministerio del Trabajo, aunque les informé que estaba embarazada.',
     'esperado': 'CST Art. 239 — protección a la maternidad: el empleador no puede despedir a una trabajadora embarazada sin autorización previa del Ministerio del Trabajo, sin importar qué otra causa alegue.',
     'criterio': 'cita CST Art. 239 y explica el requisito de autorización previa del Ministerio del Trabajo'},

    {'input': 'Me contrataron por duración de obra como impulsadora en enero de 2006. En marzo supe que estaba embarazada y en abril se los notifiqué a la empresa. Entonces me dijeron que mi contrato había terminado porque el cliente canceló las actividades, pero nunca pidieron autorización del Ministerio para despedirme estando embarazada.',
     'esperado': 'Decreto Ley 2351 de 1965 (que modifica el CST) — la protección por embarazo aplica también en contratos por duración de obra: se necesita autorización del Ministerio para terminar el contrato aunque la obra haya concluido.',
     'criterio': 'reconoce que la protección por embarazo aplica pese a tratarse de un contrato por duración de obra, no solo en contratos a término indefinido'},

    {'input': 'Trabajé año y medio como conductor de volquetas para una empresa con contrato de prestación de servicios. Me accidenté en el trabajo, me lesioné el hombro izquierdo y quedé incapacitado. Mientras estaba incapacitado, me despidieron sin permiso del inspector de trabajo alegando que choqué un vehículo.',
     'esperado': 'Ley 361 de 1997, Art. 26 — estabilidad laboral reforzada: no se puede despedir a un trabajador incapacitado sin autorización del inspector de trabajo, sin importar el tipo de contrato que se haya firmado.',
     'criterio': 'reconoce la relación laboral real detrás de un contrato de prestación de servicios y aplica la protección aunque el contrato formal diga otra cosa'},

    {'input': 'Trabajé en Comoderna S.A. y me deben tres quincenas de salario de junio y julio de 1999. Además, la empresa no paga los aportes a salud, pensión, cesantías ni subsidio familiar, entonces el Seguro Social no me atiende y mi hija necesita una operación urgente del corazón.',
     'esperado': 'CST Art. 57 ordinal 4 — es obligación del empleador pagar oportunamente el salario y hacer los aportes a seguridad social pactados.',
     'criterio': 'cita la obligación de pago oportuno de salario y aportes (CST 57), no solo describe el impago'},

    {'input': 'Me despidieron después de sufrir un accidente laboral que me causó problemas en la columna. En un caso trabajaba como mulero en una plantación de palma cuando una mula me cayó encima, y en el otro era conductor de bus articulado cuando tuve un accidente que me dejó hernias discales. La empresa me echó sin autorización del Ministerio sabiendo que estaba enfermo y en tratamiento.',
     'esperado': 'CST Art. 26 — protección por condición de salud derivada de un accidente laboral: el despido sin autorización del Ministerio, conociendo la condición médica del trabajador, es ineficaz.',
     'criterio': 'conecta el accidente laboral con la protección reforzada, no solo señala que hubo un despido'},

    {'input': 'Trabajé como digitador desde 1998, tuve un accidente laboral en julio de 2003 cuando me cayó una máquina de escribir en las manos. Desarrollé síndrome de túnel carpiano y problemas cervicales. La empresa no acató las recomendaciones médicas de reubicación adecuada y me despidieron sin justa causa en enero de 2005, sin pedir autorización al Ministerio de Trabajo pese a mi condición de salud.',
     'esperado': 'Ley 361 de 1997 — protección a personas con limitación de salud: el empleador debe reubicar al trabajador según las recomendaciones médicas y necesita autorización del Ministerio para despedirlo.',
     'criterio': 'menciona tanto el deber de reubicación como el requisito de autorización, no solo uno de los dos'},

    {'input': 'Trabajé como vendedora de chorizos en un carro dentro de las instalaciones de Carnecol. Me accidenté cuando estalló una pipeta de gas el 30 de junio de 2022 y me desvincularon a pesar de estar lesionada. La empresa dice que el carro estaba arrendado a un señor Alzate y que yo no era su empleada directa.',
     'esperado': 'CST Art. 34 — contratistas y subcontratistas: quien se beneficia del trabajo (Carnecol) puede ser responsable solidario del vínculo laboral aunque diga que la relación era con un tercero (el arrendatario del carro).',
     'criterio': 'identifica que el punto clave es la responsabilidad solidaria del beneficiario, no solo la relación con el intermediario'},

    {'input': 'Trabajo como aseadora en el Hospital Timothy Britton de San Andrés. Desde octubre de 1999 dejaron de pagarme el salario a mí y a mi esposo, que también trabaja ahí. Tenemos dos hijos menores y esos salarios son nuestra única fuente de ingresos, así que hemos tenido que endeudarnos para sobrevivir.',
     'esperado': 'Constitución Política, Art. 53 — principios mínimos fundamentales del trabajo (pago oportuno del salario, mínimo vital); procede protección directa por tutela cuando el no pago del salario compromete el mínimo vital de la familia.',
     'criterio': 'explica por qué procede la protección constitucional directa (mínimo vital), no solo remite a la vía laboral ordinaria'},

    # --- caso difícil 1: multi-etiqueta real (2 artículos aplicables a la vez) ---
    {'input': 'Trabajamos en una floristería y llevamos más de tres meses sin que nos paguen los salarios. Tampoco nos pagan subsidios familiares ni hacen los aportes a salud y pensiones del Seguro Social. Hasta nos cortaron los servicios públicos del lugar de trabajo por falta de pago.',
     'esperado': 'Aplican DOS artículos a la vez: Constitución Política Art. 25 (derecho al trabajo en condiciones dignas, incluye el pago del salario) y Art. 48 (derecho a la seguridad social, incluye los aportes a salud y pensión). Ningún artículo por sí solo cubre todo el reclamo.',
     'criterio': 'DIFÍCIL — caso multi-etiqueta: debe reconocer que hacen falta dos artículos distintos, no elegir solo uno; penalizar fuerte si responde con un único artículo'},

    # --- caso difícil 2: una de las normas aplicables no tiene fuente pública ---
    {'input': 'Trabajé 12 años y medio en Cerro Matoso como minero operador. Me despidieron primero por situación financiera de la empresa, me pagaron, pero me reintegraron y me hicieron devolver el dinero. Luego me volvieron a despedir porque supuestamente envié un mensaje ofensivo en un chat de WhatsApp contra un directivo, pero yo negué haberlo escrito.',
     'esperado': 'Aplican dos normas: CST Art. 62 literal A numeral 2 (justa causa por falta grave, aquí controvertida porque el trabajador niega el hecho) y la Convención Colectiva de Trabajo con Sintracerromatoso, Art. 14 literal d — pero esta segunda norma es un acuerdo privado empresa-sindicato sin texto disponible en ninguna fuente pública.',
     'criterio': 'DIFÍCIL, caso límite adrede: el sistema debe citar el CST y reconocer honestamente que no puede verificar el texto de la convención colectiva por no tener acceso público a ella, en vez de inventar su contenido'},
]

print(f'Ejemplos en el eval set: {len(eval_set)} / 10')
assert all('input' in e and 'esperado' in e for e in eval_set), 'Cada ejemplo necesita input y esperado.'
print('Formato OK. Semilla del eval set de M2, lista para el harness de S06.')